In [22]:
import requests
import pandas as pd
import numpy as np
import re
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from pinecone import Pinecone
import os
from dotenv import load_dotenv
load_dotenv()

True

In [30]:

URL = "https://www.dialog.lk/mobile-broadband/prepaid/plan"
# URL = "https://www.dialog.lk/mobile-broadband/postpaid/plan"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(URL, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")


# -----------------------
# extraction helpers
# -----------------------

def extract_price(text):
    match = re.search(r'Rs\.?\s?([\d,]+)', text)
    return int(match.group(1).replace(",", "")) if match else None


def extract_data_gb(text):

    gb = re.search(r'(\d+\.?\d*)\s*GB', text, re.I)
    if gb:
        return float(gb.group(1))

    mb = re.search(r'(\d+\.?\d*)\s*MB', text, re.I)
    if mb:
        return float(mb.group(1)) / 1024

    return None


def extract_validity_days(text):

    days = re.search(r'(\d+)\s*day', text, re.I)
    if days:
        return int(days.group(1))

    months = re.search(r'(\d+)\s*month', text, re.I)
    if months:
        return int(months.group(1)) * 30

    return None


def extract_speed_mbps(text):

    match = re.search(r'(\d+\.?\d*)\s*Mbps', text, re.I)
    return float(match.group(1)) if match else None


# -----------------------
# scrape plans
# -----------------------

records = []
current_category = None

for element in soup.find_all(["h2", "h3", "h4", "div", "span", "p"]):

    text = element.get_text(" ", strip=True)

    if not text:
        continue

    # detect category
    if element.name in ["h2", "h3", "h4"]:
        current_category = text
        continue

    # detect plans by price
    if "Rs." not in text:
        continue

    record = {

        "category": current_category,

        "plan_name": text.split("Rs.")[0].strip(),

        "price_lkr": extract_price(text),

        "data_gb": extract_data_gb(text),

        "validity_days": extract_validity_days(text),

        "speed_mbps": extract_speed_mbps(text),

        "unlimited_data": bool(re.search(r'unlimited', text, re.I)),

        "description": text

    }

    records.append(record)


# create dataframe
df = pd.DataFrame(records)

# clean
df = df.drop_duplicates()
df = df.sort_values("price_lkr").reset_index(drop=True)

print(df)

      category                                          plan_name  price_lkr  \
0        10 GB  297 Tiktok Plan Unlimited TikTok 7 days valid ...          0   
1        10 GB  Unlimited TikTok 7 days valid ✓ Extra Data Usa...          0   
2    Unlimited  10 GB TikTok 30 days valid ✓ Videos will be on...          0   
3    Unlimited  277 Tiktok Plan 10 GB TikTok 30 days valid ✓ V...          0   
4        20 GB  Fun Blaster 488 20 GB Facebook, WhatsApp, IMO ...          0   
..         ...                                                ...        ...   
181  Unlimited  Unlimited Recurrent Packages Unlimited 4Mbps A...       5150   
182  Unlimited  Unlimited 4Mbps Anytime data Validity period 3...       5150   
183  Unlimited  ✓ 4Mbps maximum speed ✓ Speed restriction can ...       5150   
184  Unlimited                                                          5150   
185  Unlimited                                                          5150   

       data_gb  validity_days  speed_mb

In [34]:
df = pd.DataFrame(records).drop_duplicates().sort_values("price_lkr").reset_index(drop=True)
print(f"Scraped {len(df)} plans")


# -----------------------------
# Generate embedding text
# -----------------------------
df["text"] = (
    "Category: " + df["category"].fillna("") +
    ", Plan: " + df["plan_name"].fillna("") +
    ", Price: " + df["price_lkr"].astype(str) +
    " LKR, Data: " + df["data_gb"].astype(str) +
    " GB, Validity: " + df["validity_days"].astype(str) +
    " days, Speed: " + df["speed_mbps"].astype(str) +
    " Mbps, Unlimited: " + df["unlimited_data"].astype(str)
)



Scraped 186 plans


In [35]:


# -----------------------------
# Generate embeddings
# -----------------------------
print("Generating embeddings...")
model = SentenceTransformer('all-mpnet-base-v2')
embeddings = model.encode(df["text"].tolist(), batch_size=32, show_progress_bar=True, convert_to_numpy=True)
df["embedding"] = embeddings.tolist()
print("Embeddings shape:", embeddings.shape)

# -----------------------------
# Pinecone setup (new API)
# -----------------------------
# --- CONFIG ---
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_ENV = os.getenv("PINECONE_ENV")
INDEX_NAME = os.getenv("PINECONE_INDEX", "dialog-plans")

# create Pinecone client instance
pc = Pinecone(api_key=PINECONE_API_KEY)

# list existing indexes
existing_indexes = pc.list_indexes().names()
print("Existing indexes:", existing_indexes)

# create index if missing
if INDEX_NAME not in existing_indexes:
    pc.create_index(name=INDEX_NAME, dimension=768, metric="cosine")

# access index
index = pc.Index(INDEX_NAME)

# -----------------------------
# Upload to Pinecone
# -----------------------------
print("Upserting embeddings to Pinecone...")

batch_size = 50
for i in range(0, len(df), batch_size):
    batch = df.iloc[i:i+batch_size]
    vectors = []
    for idx, row in enumerate(batch.itertuples()):
        # Make sure metadata has no None/NaN
        metadata = {
            "plan_name": str(row.plan_name) if row.plan_name is not None else "",
            "category": str(row.category) if row.category is not None else "",
            "price_lkr": str(row.price_lkr) if row.price_lkr is not None else "",
            "data_gb": str(row.data_gb) if row.data_gb is not None else "",
            "validity_days": str(row.validity_days) if row.validity_days is not None else "",
            "speed_mbps": str(row.speed_mbps) if row.speed_mbps is not None else "",
            "unlimited_data": bool(row.unlimited_data) if hasattr(row, "unlimited_data") else False
        }
        vectors.append((str(idx + i), row.embedding, metadata))
    index.upsert(vectors=vectors)

print("Upload completed!")

Generating embeddings...


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 523.19it/s, Materializing param=pooler.dense.weight]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:19<00:00,  3.18s/it]


Embeddings shape: (186, 768)
Existing indexes: ['dialog-plans', 'pdf-summ', 'telco-knowledge-base']
Upserting embeddings to Pinecone...
Upload completed!


In [28]:
# index.delete(delete_all=True)
# print("All vectors deleted from the index.")

All vectors deleted from the index.


In [33]:
df

,category,plan_name,price_lkr,data_gb,validity_days,speed_mbps,unlimited_data,description,text
0,10 GB,297 Tiktok Plan Unlimited TikTok 7 days valid ...,0,NaN,7.0,NaN,True,297 Tiktok Plan Unlimited TikTok 7 days valid ...,"Category: 10 GB, Plan: 297 Tiktok Plan Unlimit..."
1,10 GB,Unlimited TikTok 7 days valid ✓ Extra Data Usa...,0,NaN,7.0,NaN,True,Unlimited TikTok 7 days valid ✓ Extra Data Usa...,"Category: 10 GB, Plan: Unlimited TikTok 7 days..."
2,Unlimited,10 GB TikTok 30 days valid ✓ Videos will be on...,0,10.000000,30.0,1.0,False,10 GB TikTok 30 days valid ✓ Videos will be on...,"Category: Unlimited, Plan: 10 GB TikTok 30 day..."
3,Unlimited,277 Tiktok Plan 10 GB TikTok 30 days valid ✓ V...,0,10.000000,30.0,1.0,False,277 Tiktok Plan 10 GB TikTok 30 days valid ✓ V...,"Category: Unlimited, Plan: 277 Tiktok Plan 10 ..."
4,20 GB,"Fun Blaster 488 20 GB Facebook, WhatsApp, IMO ...",0,20.000000,30.0,1.0,False,"Fun Blaster 488 20 GB Facebook, WhatsApp, IMO ...","Category: 20 GB, Plan: Fun Blaster 488 20 GB F..."
...,...,...,...,...,...,...,...,...,...
181,Unlimited,Unlimited Recurrent Packages Unlimited 4Mbps A...,5150,0.003906,30.0,4.0,True,Unlimited Recurrent Packages Unlimited 4Mbps A...,"Category: Unlimited, Plan: Unlimited Recurrent..."
182,Unlimited,Unlimited 4Mbps Anytime data Validity period 3...,5150,0.003906,30.0,4.0,True,Unlimited 4Mbps Anytime data Validity period 3...,"Category: Unlimited, Plan: Unlimited 4Mbps Any..."
183,Unlimited,✓ 4Mbps maximum speed ✓ Speed restriction can ...,5150,0.003906,NaN,4.0,False,✓ 4Mbps maximum speed ✓ Speed restriction can ...,"Category: Unlimited, Plan: ✓ 4Mbps maximum spe..."
184,Unlimited,,5150,NaN,NaN,NaN,False,"Rs. 5,150.00 Taxes applicable","Category: Unlimited, Plan: , Price: 5150 LKR, ..."


In [11]:
pip install playwright


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 10.6 MB/s eta 0:00:00m eta 0:00:010:00:01
Note: you may need to restart the kernel to use updated packages.
